# 10 - Decision support, robustness, and session merge

This notebook focuses on decision support: compromise solutions, clustering, robustness checks, and merging sessions for downstream analysis.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `OptimisationSimulation.get_best_compromise_solution`
- `OptimisationSimulation.run_clustering`
- `OptimisationSimulation.run_robustness_analysis`
- `OptimisationSimulation.merge`
- `FileNotFoundError`
- `OptimisationSimulation.__new__`
- `optim.load_outputs_optimisation`
- `optim.get_best_compromise_solution`
- `optim.plot_best_compromise_solutions`
- `optim.run_clustering`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Decision support: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
from pathlib import Path

import pandas as pd
from accim.parametric_and_optimisation.main import OptimisationSimulation, ParametricSimulation


## Decision support: locate latest output files

This cell locates the most recent result files that will be reused for downstream analysis.

By resolving file paths here, the notebook stays reproducible and avoids accidentally mixing outputs from different runs.

Key arguments used in this cell and why they matter:
- `parents`: defines this setting for the current call.
- `exist_ok`: defines this setting for the current call.
- `key`: defines this setting for the current call.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `OUT_DIR`: reused in subsequent workflow blocks.
- `optim_candidates`: reused in subsequent workflow blocks.
- `param_candidates`: reused in subsequent workflow blocks.
- `optim_csv`: reused in subsequent workflow blocks.
- `param_csv`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `OUT_DIR`, `optim_candidates`, `param_candidates`.

In [ ]:

PROJECT_ROOT = Path.cwd()
OUT_DIR = PROJECT_ROOT / "nb10_decision_support_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

optim_candidates = sorted(
    PROJECT_ROOT.glob("**/outputs_optimisation_*.csv"),
    key=lambda p: p.stat().st_mtime,
)
param_candidates = sorted(
    PROJECT_ROOT.glob("**/outputs_param_simulation_*.csv"),
    key=lambda p: p.stat().st_mtime,
)

if not optim_candidates:
    raise FileNotFoundError("No optimisation outputs were found. Run notebook 07 or 08 first.")
if not param_candidates:
    raise FileNotFoundError("No parametric outputs were found. Run notebook 01, 02, 03, 05, or 06 first.")

optim_csv = optim_candidates[-1]
param_csv = param_candidates[-1]

print("Optimisation source:", optim_csv)
print("Parametric source:", param_csv)


## Decision support: run compromise, clustering, and robustness analysis

This stage converts optimization outputs into decision-support evidence.

Compromise, clustering, and robustness checks help translate Pareto solutions into actionable choices.

Key arguments used in this cell and why they matter:
- `csv_path`: path to the output CSV file to load.
- `method`: analysis method to run (for example morris or topsis).
- `weights`: preference weights for multi-criteria methods.
- `out_dir`: output folder for generated artifacts and results.
- `mcdm_configs`: configuration list used to compare MCDM methods.
- `normalize_per_m2`: whether results are normalized by floor area.

Variables created in this cell:
- `optim`: reused in subsequent workflow blocks.
- `sol_knee`: reused in subsequent workflow blocks.
- `sol_topsis`: reused in subsequent workflow blocks.
- `mcdm_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Review compromise, clustering, or robustness outputs for consistency with objective trends.
- Keep selected decision-support tables/plots for reporting and comparison.

In [ ]:

optim = OptimisationSimulation.__new__(OptimisationSimulation)
optim.last_run_type = None
optim.outputs_optimisation = None
optim.outputs_optimisation_filepath = None
optim.outputs_optimisation_hourly = None
optim.outputs_optimisation_monthly = None
optim.optimisation_csv_paths_non_dominated = []
optim.optimisation_csv_paths_dominated = []
optim.optimisation_csv_paths_non_dominated_by_epw = {}
optim.optimisation_csv_paths_dominated_by_epw = {}
optim.evaluators = {}
optim.epws = []
optim.buildings = []

optim.load_outputs_optimisation(csv_path=str(optim_csv))
print("Loaded optimisation rows:", len(optim.outputs_optimisation))

sol_knee = optim.get_best_compromise_solution(method="knee_point")
print("Knee-point compromise rows:", len(sol_knee))

sol_topsis = optim.get_best_compromise_solution(method="topsis", weights=[0.7, 0.3])
print("TOPSIS compromise rows:", len(sol_topsis))

mcdm_df = optim.plot_best_compromise_solutions(
    out_dir=str(OUT_DIR),
    mcdm_configs=[
        {"method": "knee_point"},
        {"method": "topsis"},
        {"method": "topsis", "weights": [0.7, 0.3], "label": "topsis_w70_30"},
    ],
    normalize_per_m2=False,
)
print("Compromise summary rows:", len(mcdm_df))


This stage converts optimization outputs into decision-support evidence.

Compromise, clustering, and robustness checks help translate Pareto solutions into actionable choices.

Key arguments used in this cell and why they matter:
- `n_clusters`: number of clusters for solution grouping.
- `cluster_by`: column group used for clustering.
- `pareto_only`: whether analysis is restricted to Pareto solutions.
- `out_dir`: output folder for generated artifacts and results.
- `optimal_solutions_df`: candidate solution table used for robustness analysis.
- `epws_robustness`: EPW set used for robustness checks.
- `normalize_per_m2`: whether results are normalized by floor area.

Variables created in this cell:
- `robust_epws`: reused in subsequent workflow blocks.
- `optimal_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Review compromise, clustering, or robustness outputs for consistency with objective trends.
- Keep selected decision-support tables/plots for reporting and comparison.

In [ ]:

optim.run_clustering(
    n_clusters=2,
    cluster_by="parameters",
    pareto_only=True,
    out_dir=str(OUT_DIR),
)
print("Clustering completed.")

if "epw" not in optim.outputs_optimisation.columns:
    raise ValueError("The optimisation output does not contain the 'epw' column.")

robust_epws = [str(v) for v in optim.outputs_optimisation["epw"].dropna().unique().tolist()][:2]
if not robust_epws:
    raise ValueError("No EPW labels are available to run robustness analysis.")

optimal_df = sol_topsis if not sol_topsis.empty else sol_knee
if optimal_df.empty:
    optimal_df = optim.outputs_optimisation.head(1)

optim.run_robustness_analysis(
    optimal_solutions_df=optimal_df,
    epws_robustness=robust_epws,
    out_dir=str(OUT_DIR),
    normalize_per_m2=False,
)
print("Robustness analysis completed.")


## Decision support: load parametric outputs and merge analysis sessions

This block reloads previously exported outputs to continue analysis without rerunning simulations.

Always validate schema and row counts right after loading so subsequent transformations remain reliable.

Key arguments used in this cell and why they matter:
- `csv_path`: path to the output CSV file to load.
- `inplace`: whether merge modifies the current object.

Variables created in this cell:
- `param_sim`: reused in subsequent workflow blocks.
- `merged_obj`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Validate dataframe dimensions immediately after loading.
- Check that required columns exist before transformations and plotting.

In [ ]:

param_sim = ParametricSimulation.__new__(ParametricSimulation)
param_sim.last_run_type = None
param_sim.outputs_param_simulation = None
param_sim.outputs_param_simulation_hourly = None
param_sim.outputs_param_simulation_monthly = None
param_sim.outputs_param_simulation_filepath = None
param_sim.epws = []
param_sim.buildings = []

param_sim.load_outputs_parametric(csv_path=str(param_csv))
print("Loaded parametric rows:", len(param_sim.outputs_param_simulation))

merged_obj = optim.merge([param_sim], inplace=False)
print("Merged object type:", type(merged_obj).__name__)
